In [ ]:
# 2026-05-15
from apis_core.apis_metainfo.models import Uri, Collection, CollectionType
from apis_core.apis_entities.models import Place, Institution
from apis_core.apis_relations.models import InstitutionPlace
from apis_core.apis_vocabularies.models import InstitutionPlaceRelation
from acdh_tei_pyutils.tei import TeiReader
from acdh_tei_pyutils.utils import any_xpath, get_xmlid
from normdata.utils import import_from_normdata
from tqdm import tqdm

In [ ]:
col, _ = Collection.objects.get_or_create(name="Schönberg-Briefe: Universal-Edition und Dreililien")
domain = "schoenberg-ue"
col_type, _ = CollectionType.objects.get_or_create(name="Projekt")
col.description = """
Arnold Schönberg: Briefwechsel mit den Verlagen Universal-Edition und Dreililien. Hrsg. von Katharina Bleier und Therese Muxeneder unter Mitarbeit von Jannik Franz und Philipp Kehrer, Universität für Musik und darstellende Kunst Wien und Arnold Schönberg Center, Wien
<a href="https://www.schoenberg-ue.at">https://www.schoenberg-ue.at</a>
"""
col.collection_type = col_type
col.save()

In [ ]:
source_file_uri = "https://raw.githubusercontent.com/Schoenberg-UE/Schoenberg-Data/refs/heads/main/data/indices/Organisationen.xml"

In [ ]:
doc = TeiReader(source_file_uri)

In [ ]:
items = doc.any_xpath(".//tei:org[@xml:id]")
url_stub = "https://www.schoenberg-ue.at/ue/organisations/"
item_item_relation_type = InstitutionPlaceRelation.objects.get(name="angesiedelt in") 


In [ ]:
with_id = 0
no_id = 0
print(len(items))
for x in tqdm(items, total=len(items)):
    xml_id = get_xmlid(x)
    tmp_uri = f"{url_stub}{xml_id}"
    label = any_xpath(x, "./tei:orgName[@type='reg']")[0].text
    try:
        gnd = any_xpath(x, "./tei:idno[@type='gnd']/text()")[0]
        with_id += 1
        entity = import_from_normdata(gnd, "institution")
        if entity:
            pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
            pmb_uri.entity = entity
            pmb_uri.save()
            entity.collection.add(col)
    except IndexError:
        pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
        entity = pmb_uri.entity
        if entity:
            pass
        else:
            item = {
                "name": label,
            }
            entity = Institution.objects.create(**item)
            entity.collection.add(col)
            pmb_uri.entity = entity
            pmb_uri.save()
        for y in any_xpath(x, ".//tei:placeName[@key]/@key"):
            related_xml_id = y
            related_url = f"https://www.schoenberg-ue.at/ue/places/{related_xml_id}"
            related_entity = Uri.objects.get(uri=related_url).entity
            related_entity = Place.objects.get(id=related_entity.id)
            instution = Institution.objects.get(id=entity.id)
            InstitutionPlace.objects.get_or_create(related_institution=instution, related_place=related_entity, relation_type=item_item_relation_type)
